In [ ]:
from envs import *
from agents import *
from models.model import QNetwork
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch.optim as optim
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
from collections import namedtuple, deque
import datetime


now = datetime.datetime.now()   
curr_time = now.strftime("%H_%M_%S")
curr_date = now.strftime("%d_%m_%Y")
curr_date_time = str('d_')+curr_date+str('_t_')+curr_time



try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False


log_wandb = False

## Experiments

In [ ]:
# read a csv file
expt_df = pd.read_csv('experiments_all.csv')
expt_df.head()



In [ ]:
num_experiments = len(expt_df)


for i in range(num_experiments):
    expt_setup = expt_df.iloc[i].to_dict()
    expt_seeds = [expt_setup['SEED-1'], 
                  expt_setup['SEED-2'], 
                  expt_setup['SEED-3'], 
                  expt_setup['SEED-4'], 
                  expt_setup['SEED-5']]

    for seed in expt_seeds:

        EXPERIMENT_PARAMS = {
            "DELAY": expt_setup["DELAY"],
            "TOXITY_PENALTY": expt_setup["TOXICITY_PENALTY"],
            "TARGETS": expt_setup["TARGETS"],
            "DELAY_MEAN": expt_setup["DELAY_MEAN"],
            "DELAY_STD": expt_setup["DELAY_STD"],
            "SEED": seed,
            "AGENT_NAME": "DDQNAgent",
            "Description": "Mention phase of the experiment here",
            "RUN_NUM": expt_setup["RUN_NUM"],
        }


        NUM_EPISODES = 5000
        PROJECT_NAME = "FOOD ENV NEW TEST EXPERIMENTS"
        RUN_NAME = (
            EXPERIMENT_PARAMS["AGENT_NAME"]
            + "_Run_" + str(EXPERIMENT_PARAMS["RUN_NUM"])
            + "_Seed_" + str(EXPERIMENT_PARAMS["SEED"])
        )


        if EXPERIMENT_PARAMS["DELAY"] == 0:
            use_delay = False
        else:
            use_delay = True
        
        if EXPERIMENT_PARAMS["TOXITY_PENALTY"] == 0:
            toxicity_scale = 0.0
        else:
            toxicity_scale = 2.0 #5.0
        
        if EXPERIMENT_PARAMS["TARGETS"] == 0:
            target_nutrients = ["Carbs"]
        else:
            target_nutrients = ["Carbs", "Protein", "Fat"]
        
        if EXPERIMENT_PARAMS["DELAY_MEAN"] == 0 and EXPERIMENT_PARAMS["DELAY_STD"] == 0:
            file_path = 'foods_dataset/food_data_low_low_delay_std.csv'
        elif EXPERIMENT_PARAMS["DELAY_MEAN"] == 0 and EXPERIMENT_PARAMS["DELAY_STD"] == 1:
            file_path = 'foods_dataset/food_data_low_high_delay_std.csv'
        elif EXPERIMENT_PARAMS["DELAY_MEAN"] == 1 and EXPERIMENT_PARAMS["DELAY_STD"] == 0:
            file_path = 'foods_dataset/food_data_medium_low_delay_std.csv'
        elif EXPERIMENT_PARAMS["DELAY_MEAN"] == 1 and EXPERIMENT_PARAMS["DELAY_STD"] == 1:
            file_path = 'foods_dataset/food_data_medium_high_delay_std.csv'
        elif EXPERIMENT_PARAMS["DELAY_MEAN"] == 2 and EXPERIMENT_PARAMS["DELAY_STD"] == 0:
            file_path = 'foods_dataset/food_data_long_low_delay_std.csv'
        elif EXPERIMENT_PARAMS["DELAY_MEAN"] == 2 and EXPERIMENT_PARAMS["DELAY_STD"] == 1:
            file_path = 'foods_dataset/food_data_long_high_delay_std.csv'
        elif EXPERIMENT_PARAMS["DELAY_MEAN"] == 3 and EXPERIMENT_PARAMS["DELAY_STD"] == 0:
            file_path = 'foods_dataset/food_data_verylong_low_delay_std.csv'
        elif EXPERIMENT_PARAMS["DELAY_MEAN"] == 3 and EXPERIMENT_PARAMS["DELAY_STD"] == 1:
            file_path = 'foods_dataset/food_data_verylong_high_delay_std.csv'
        
        AGENT_NAME = EXPERIMENT_PARAMS["AGENT_NAME"]#'MCAgent' 
        
        
        FILE_PATH = file_path
        
        
        SEED = EXPERIMENT_PARAMS["SEED"]
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        
        
        df = pd.read_csv(FILE_PATH)
        
        
        ENV_AGRS = {"max_steps":50,
                      "normalise":True,
                      "one_hot_embedding":True,
                      "embed_size":None,
                      "target_loc":None,
                      "use_delay": use_delay,
                      "target_nutrients": target_nutrients,
                      "toxicity_scale": toxicity_scale
                      }
        
        
        env = NutriRL(file_path=FILE_PATH, **ENV_AGRS)
        env.reset()
        
        
        AGENT_ARGS = {"gamma": 0.99,
                      "n_step": 5,
                      "max_q_clip": 500.0,
                      "max_grad_norm": 10.0,}
        
        TRAINING_PARAMS = {
                            "num_episodes": NUM_EPISODES,
                            "min_replay_size": 10_000,
                            "batch_size": 128,
                            "train_every": 8,
                            "target_update_every": 8000,
                            "eps_start": 1.0,
                            "eps_end": 0.01,
                            "eps_decay_steps": 200_000,
                            "lr_start": 1e-3,
                            "lr_end": 1e-4,
                          }
        
        display(df.head())
        
        if log_wandb:
            print(f'Running project {PROJECT_NAME}, run {RUN_NAME} on wandb at curr date and time {curr_date_time}')
            wandb.init(
                    project=PROJECT_NAME,
                    name=RUN_NAME,
                    config={
                        "experiment_params": EXPERIMENT_PARAMS,
                        "agent": AGENT_NAME,
                        "agent_args": AGENT_ARGS,
                        "training_params": TRAINING_PARAMS,
                        "env_args": ENV_AGRS,
                        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
                        "file_path": FILE_PATH,
                        "datetime": curr_date_time  
                    })
            
        agent_class = globals()[AGENT_NAME]
        agent = agent_class(env) #, **AGENT_ARGS
        
        scores, picks = agent.train(log_wandb=log_wandb, printing=True, **TRAINING_PARAMS)
        
        if not os.path.exists("./saved_models/"+AGENT_NAME):
            os.makedirs("./saved_models/"+AGENT_NAME)
        path = "./saved_models/"+AGENT_NAME+"/" + RUN_NAME
        agent.save_model(path, log_wandb=log_wandb)
        
        agent.load_model(path)
        agent_memory = agent.generate_episode(log_wandb = log_wandb, episode_idx=0)
        df = agent.infer_episode(agent_memory)
        display(df)
        
        